# Sentinel - train every model on a free Colab GPU

Runtime > Change runtime type > **T4 GPU**. Every step is keyless. The same scripts run locally (`make train`); this notebook only adds the Colab setup. Outputs land in `models/` and `models/onnx/`.

In [ ]:
!git clone https://github.com/yugh88/VIT_YughJuneja_Hackathon.git
%cd VIT_YughJuneja_Hackathon
!pip install -q -r requirements.txt -r requirements-train.txt
import os; os.environ['PYTHONPATH'] = os.getcwd() + '/src'

## 1. Data (public, keyless)
PhraseBank, twitter-financial-news sentiment + topic, FiQA-2018, SEntFiN; then weak event labels from historical GDELT windows.

In [ ]:
!python -m training.download_data training
!python -m training.weak_labels --windows 700

## 2. Translate-train data with NLLB-200 (8 languages)

In [ ]:
!python -m training.translate phrasebank
!python -m training.translate events --per-lang 1200

## 3. Fine-tune (3 epochs each)

In [ ]:
!python -m training.train_sentiment en
!python -m training.train_sentiment multi
!python -m training.train_events

## 4. Export to ONNX + INT8, then evaluate

In [ ]:
!python -m training.export_onnx sentiment-en sentiment-multi event-multi --zero-shot
!python -m training.evaluate